# 04 — Cashiers EDA
## Staff Roster, Shift Patterns & Per-Cashier Activity

**Purpose.** Analysis of the staff roster (`cashiers.csv`) and the
per-cashier activity patterns derived from the transactions file.

**This is the smallest of the four EDA notebooks** because the cashiers file
itself only has 12 rows. The interesting analysis comes from joining cashiers
with transactions to see how each staff member operates.

**What this notebook covers:**
- Cashier roster: shifts and the suspicious flag
- Activity per cashier (volume, hours worked, sections served)
- Relationship between cashier shifts and intra-day order patterns
- Cross-reference with anomalies (the C07 deep dive complements
  `03_anomalies_eda.ipynb`)

**Other notebooks in this series:**
- `01_transactions_eda.ipynb` — main transactions
- `02_customers_eda.ipynb` — customer profiles
- `03_anomalies_eda.ipynb` — anomaly labels & fraud signals

---

## Notebook structure
1. Setup and data loading
2. Cashier roster
3. Per-cashier activity volumes
4. Shift patterns and intra-day distribution
5. Section coverage by cashier
6. Cross-reference with anomalies
7. Summary


## 1. Setup and data loading

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# === ADJUST THIS PATH IF YOUR FOLDER LAYOUT IS DIFFERENT ===
DATASETS_DIR = '../dataset_generation/datasets'

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['font.size'] = 10

print(f"Working directory: {os.getcwd()}")
print(f"Looking for data in: {os.path.abspath(DATASETS_DIR)}")

In [ ]:
# Load cashiers, anomalies, and transactions
cashiers = pd.read_csv(os.path.join(DATASETS_DIR, 'cashiers.csv'))
anomalies = pd.read_csv(os.path.join(DATASETS_DIR, 'anomalies_ground_truth.csv'))
df = pd.read_csv(os.path.join(DATASETS_DIR, 'enterprise_pos_dataset.csv'), sep='|')
df['order_date'] = pd.to_datetime(df['order_date'])

anom_oids = set(anomalies['order_id'])

print(f"Cashiers loaded:    {len(cashiers)} staff")
print(f"Anomalies loaded:   {len(anomalies)} labels")
print(f"Transactions:       {len(df):,} line items in {df['order_id'].nunique():,} orders")

## 2. Cashier roster

12 cashiers across 3 shifts. One of them (C07) is flagged as the suspicious
actor — this is ground truth for the Module 3 evaluation.

In [ ]:
print("Full roster:")
cashiers

In [ ]:
# Shift breakdown
shift_counts = cashiers['shift'].value_counts()
print("\nCashiers per shift:")
print(shift_counts)

print(f"\nFlagged cashier(s): {cashiers[cashiers['flagged']]['cashier_id'].tolist()}")

## 3. Per-cashier activity volumes

How busy is each cashier? Some process many orders, others fewer — this is
useful baseline data for the anomaly detector.

In [ ]:
orders_unique = df.drop_duplicates('order_id')
cashier_activity = orders_unique['cashier_id'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(11, 5))
colors = ['crimson' if cid == 'C07' else 'steelblue' for cid in cashier_activity.index]
bars = ax.bar(cashier_activity.index, cashier_activity.values,
              color=colors, edgecolor='black')
ax.set_title('Total orders processed per cashier (18 months)')
ax.set_ylabel('Number of orders')
ax.set_xlabel('Cashier ID')
for bar, val in zip(bars, cashier_activity.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{val:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print(f"\nMean orders per cashier:   {cashier_activity.mean():.0f}")
print(f"Median orders per cashier: {cashier_activity.median():.0f}")
print(f"Min:                       {cashier_activity.min()}")
print(f"Max:                       {cashier_activity.max()}")

**Reading.** Cashier activity is roughly comparable across the team —
volume alone is NOT a reliable anomaly signal. C07 is not unusually busy or
quiet; what makes them suspicious is the *type* of orders they process, not
the count.

## 4. Shift patterns and intra-day distribution

Each cashier works a particular shift (morning / afternoon / evening) and
should appear primarily during those hours. Let's verify and visualize.

In [ ]:
# Convert order_time to a numeric hour
def time_to_hour(t):
    if pd.isna(t):
        return np.nan
    try:
        h = int(t[:2])
        m = int(t[3:5])
        ampm = t[-2:]
        if ampm == 'AM':
            return h + m/60 if h != 12 else m/60
        else:
            return h + m/60 if h == 12 else h + 12 + m/60
    except:
        return np.nan

orders_unique = orders_unique.copy()
orders_unique['hour_float'] = orders_unique['order_time'].apply(time_to_hour)
print("Time conversion done.")
print(orders_unique[['order_time', 'hour_float']].head())

In [ ]:
# Heatmap: cashier × hour activity
hour_bins = sorted(orders_unique['hour_float'].dropna().unique())
heatmap_data = orders_unique.groupby(['cashier_id', 'hour_float']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(heatmap_data, cmap='YlOrRd', linewidths=0.3,
            cbar_kws={'label': '# orders'}, ax=ax)
ax.set_title('Activity heatmap: cashier × hour of day')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Cashier ID')
plt.tight_layout()
plt.show()

**Reading.** Each cashier has a clear "active band" of hours
corresponding to their shift:
- **Morning shift (C01-C04)** is active 7am-2pm
- **Afternoon shift (C05-C08)** is active 12pm-8pm
- **Evening shift (C09-C12)** is active 5pm-10pm

This shift structure is itself a feature for anomaly detection: an order
processed by a morning cashier at 9pm would be highly suspicious.

## 5. Section coverage by cashier

Different cashiers serve different sections depending on their shift. The
morning shift handles Café and lunch sections; the evening shift handles
dinner sections.

In [ ]:
section_by_cashier = orders_unique.groupby(['cashier_id', 'restaurant_type']).size().unstack(fill_value=0)
section_pct = section_by_cashier.div(section_by_cashier.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(13, 6))
section_pct.plot(kind='bar', stacked=True, ax=ax,
                 colormap='tab20', edgecolor='white', width=0.85)
ax.set_title('Section coverage per cashier (% of their orders)')
ax.set_xlabel('Cashier ID')
ax.set_ylabel('% of orders')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Reading.** Morning cashiers (C01-C04) almost exclusively serve Cafe
and Healthy_Vegan (lunch). Evening cashiers serve dinner sections (Italian,
Mexican, Japanese, Steakhouse). This natural specialization is another
feature axis: a cashier processing an order outside their normal section
mix could be a flag.

## 6. Cross-reference with anomalies

We already saw in `03_anomalies_eda.ipynb` that C07 has a ~10× anomaly rate
compared to the team average. Let's add some additional cuts.

### 6.1 Anomaly rate per cashier with shift overlay

In [ ]:
cashier_stats = []
for cid in sorted(orders_unique['cashier_id'].unique()):
    sub = orders_unique[orders_unique['cashier_id'] == cid]
    n_total = len(sub)
    n_anom = sub['order_id'].isin(anom_oids).sum()
    rate = n_anom / n_total * 100 if n_total > 0 else 0
    shift = cashiers[cashiers['cashier_id'] == cid]['shift'].values[0]
    cashier_stats.append({
        'cashier_id': cid,
        'shift': shift,
        'total_orders': n_total,
        'anomaly_count': n_anom,
        'anomaly_rate_pct': rate,
    })

cs = pd.DataFrame(cashier_stats).sort_values(['shift', 'cashier_id'])

fig, ax = plt.subplots(figsize=(12, 5))
shift_colors = {'morning': '#88aa44', 'afternoon': '#eeaa22', 'evening': '#3366aa'}
bar_colors = []
for _, row in cs.iterrows():
    if row['cashier_id'] == 'C07':
        bar_colors.append('crimson')
    else:
        bar_colors.append(shift_colors[row['shift']])

bars = ax.bar(cs['cashier_id'], cs['anomaly_rate_pct'],
              color=bar_colors, edgecolor='black')
ax.set_title('Anomaly rate per cashier — colored by shift, C07 highlighted')
ax.set_ylabel('Anomaly rate (%)')
ax.set_xlabel('Cashier ID')

# Custom legend
from matplotlib.patches import Patch
legend_items = [
    Patch(facecolor='#88aa44', label='Morning shift'),
    Patch(facecolor='#eeaa22', label='Afternoon shift'),
    Patch(facecolor='#3366aa', label='Evening shift'),
    Patch(facecolor='crimson', label='C07 (flagged)'),
]
ax.legend(handles=legend_items, loc='upper right')

for bar, val in zip(bars, cs['anomaly_rate_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}%', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nPer-cashier anomaly stats:")
print(cs.to_string(index=False))

**Reading.** C07 stands out regardless of shift context. Other afternoon
cashiers (C05, C06, C08) have anomaly rates near the global mean (~0.5%),
confirming that C07's high rate is *individual* and not driven by the shift
they happen to work.

### 6.2 Time-of-day distribution of C07's anomalies

In [ ]:
c07_orders = orders_unique[orders_unique['cashier_id'] == 'C07'].copy()
c07_normal = c07_orders[~c07_orders['order_id'].isin(anom_oids)]
c07_anom = c07_orders[c07_orders['order_id'].isin(anom_oids)]

fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(c07_normal['hour_float'].dropna(), bins=20, color='steelblue',
        alpha=0.7, label='C07 normal orders', edgecolor='black')
ax.hist(c07_anom['hour_float'].dropna(), bins=20, color='crimson',
        alpha=0.7, label='C07 anomalous orders', edgecolor='black')
ax.set_title('When does C07\'s fraud happen?')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\n  C07 total orders:     {len(c07_orders)}")
print(f"  C07 normal:           {len(c07_normal)}")
print(f"  C07 anomalous:        {len(c07_anom)}")
print(f"  C07 anomaly rate:     {len(c07_anom)/len(c07_orders)*100:.2f}%")

**Reading.** C07's anomalies are concentrated near the **end of their
shift** — this is the classic `shift_end_void_cluster` fraud pattern (a
cashier voiding orders near closing to skim cash from the till). The
temporal concentration is itself a strong feature: if anomalies cluster in
specific hours rather than being uniformly distributed, that's evidence of
deliberate behavior, not random error.

## 7. Summary

### What's in the cashier data
- **12 cashiers** organized into 3 shifts (4 per shift)
- **One flagged cashier (C07)** as ground truth for Module 3 evaluation

### Key findings
1. **Shift structure is strong** — each cashier has a clear active hour band
2. **Section specialization** — morning shift handles café/lunch, evening
   shift handles dinner sections
3. **Activity volume is comparable** across cashiers — volume alone is
   not a fraud signal
4. **C07 stands out only on anomaly rate**, not on volume or activity
5. **C07's fraud clusters at end-of-shift**, suggesting deliberate behavior

### Implications for Module 3 (Anomaly Detection)
The cashier dimension provides several engineered features:
- `cashier_id` (categorical or one-hot)
- `cashier_shift` (morning/afternoon/evening)
- `is_within_normal_hours_for_cashier` (binary)
- `is_within_normal_section_for_cashier` (binary)
- `cashier_anomaly_lift` (historical anomaly rate per cashier — leakage-safe
  if computed on training data only)

These features should significantly boost a detector's F1 score over a model
that ignores the staff dimension.

---

**This is the last EDA notebook.** With all four notebooks complete, you have
full coverage of the dataset across all five PFE modules. Next steps:
- Run Module 1's `01_fp_growth_v5.ipynb` (already built) on the v5 data
- Build Module 2's forecasting notebooks
- Build Modules 3, 4, 5 in turn
